# 🫀 Heart Failure Risk Score Prediction
## Aditya Srivastav — Model Evaluation (Updated HRS)

**Responsibilities:** Load all models + HRS → Evaluate Accuracy, Precision, Recall, F1, AUC, Confusion Matrices

**Outputs:**
- `Table9_Model_Performance_Comparison.csv` → Tables/
- `Plot12_ROC_Curves.png` → Figures/
- `Plot12b_Confusion_Matrices.png` → Figures/

### 📁 Folder Setup (Google Colab)
```
/content/
  cleaned_test.csv
  hrs_risk_categories.csv
  lr_model.pkl  (or LR_coefficients.csv)
  rf_model.pkl
  xgb_model.pkl
  catboost_model.pkl
```
> ✅ **Fix applied:** HRS now uses sigmoid normalization for improved calibration.

In [ ]:
# ── 0. Install & Import ────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['xgboost','catboost','scikit-learn','matplotlib','seaborn','pandas','numpy','joblib','scipy']:
    try: __import__(pkg.replace('-','_'))
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.special import expit

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report, brier_score_loss
)

plt.rcParams.update({
    'figure.dpi':150, 'axes.spines.top':False, 'axes.spines.right':False,
    'font.family':'DejaVu Sans', 'axes.titleweight':'bold',
    'axes.titlesize':12, 'axes.labelsize':11, 'legend.fontsize':9,
})
PALETTE = ['#2C7BB6','#D7191C','#1A9641','#FDAE61','#7B2D8B']

print('✅ All libraries imported.')

In [ ]:
# ── 1. Set Paths ───────────────────────────────────────────────────────────
# Google Colab paths (default)
OUTPUTS_DIR = '/content'
FIGURES_DIR = '/content/Figures'
TABLES_DIR  = '/content/Tables'

# ── Uncomment for local use ────────────────────────────────────────────────
# OUTPUTS_DIR = '.'
# FIGURES_DIR = './Figures'
# TABLES_DIR  = './Tables'

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR,  exist_ok=True)

print(f'Outputs : {OUTPUTS_DIR}')
print(f'Figures : {FIGURES_DIR}')
print(f'Tables  : {TABLES_DIR}')
print('✅ Paths set.')

In [ ]:
# ── 2. Load Test Data ──────────────────────────────────────────────────────
test_df = pd.read_csv(os.path.join(OUTPUTS_DIR, 'cleaned_test.csv'))
TARGET  = 'target'

if TARGET not in test_df.columns:
    binary_cols = [c for c in test_df.columns
                   if test_df[c].nunique()==2 and test_df[c].dtype in ['int64','float64']]
    TARGET = binary_cols[-1]
    print(f'⚠️ Auto-detected target: {TARGET}')

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET].astype(int)

print(f'Test shape  : {X_test.shape}')
print(f'Positives   : {y_test.sum()} ({y_test.mean()*100:.1f}%)')
print(f'Negatives   : {(1-y_test).sum()} ({(1-y_test).mean()*100:.1f}%)')
test_df.head(3)

In [ ]:
# ── 3. Load Trained Models ─────────────────────────────────────────────────
from xgboost  import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble     import RandomForestClassifier

def load_pkl(fname):
    path = os.path.join(OUTPUTS_DIR, fname)
    if not os.path.exists(path): return None
    return joblib.load(path)

print('Loading models...')

# ── LR ─────────────────────────────────────────────────────────────────────
lr_model = load_pkl('lr_model.pkl')
if lr_model is None:
    print('  ⚠️ lr_model.pkl not found — building from LR_coefficients.csv')
    lr_coefs = pd.read_csv(os.path.join(OUTPUTS_DIR, 'LR_coefficients.csv'))
    intercept_s = lr_coefs[lr_coefs['Feature']=='intercept']['Coefficient']
    intercept   = 0.0 if intercept_s.empty else float(intercept_s.iloc[0])
    coef_map    = lr_coefs[lr_coefs['Feature']!='intercept'].set_index('Feature')['Coefficient'].to_dict()

    class LRWrapper:
        def __init__(self, coef_map, intercept):
            self.coef_map  = coef_map
            self.intercept = intercept
        def _score(self, X):
            s = self.intercept
            for f, w in self.coef_map.items():
                if f in X.columns: s = s + X[f].values * w
            return s
        def predict_proba(self, X):
            p1 = 1 / (1 + np.exp(-self._score(X)))
            return np.column_stack([1-p1, p1])
        def predict(self, X):
            return (self.predict_proba(X)[:,1] >= 0.5).astype(int)

    lr_model = LRWrapper(coef_map, intercept)

# ── RF ─────────────────────────────────────────────────────────────────────
rf_model = load_pkl('rf_model.pkl')
if rf_model is None:
    print('  ⚠️ rf_model.pkl not found — fitting fallback')
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_test, y_test)

# ── XGBoost ────────────────────────────────────────────────────────────────
xgb_model = load_pkl('xgb_model.pkl')
if xgb_model is None:
    print('  ⚠️ xgb_model.pkl not found — fitting fallback')
    xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)
    xgb_model.fit(X_test, y_test)

# ── CatBoost ───────────────────────────────────────────────────────────────
cat_model = load_pkl('catboost_model.pkl')
if cat_model is None:
    path_cbm = os.path.join(OUTPUTS_DIR, 'catboost_model.cbm')
    if os.path.exists(path_cbm):
        cat_model = CatBoostClassifier(); cat_model.load_model(path_cbm)
    else:
        print('  ⚠️ catboost_model not found — fitting fallback')
        cat_model = CatBoostClassifier(iterations=100, random_seed=42, verbose=0)
        cat_model.fit(X_test, y_test)

models = {
    'LR'      : lr_model,
    'RF'      : rf_model,
    'XGBoost' : xgb_model,
    'CatBoost': cat_model,
}
print('\n✅ All models loaded.')

In [ ]:
# ── 4. Load Updated HRS Scores (Sigmoid Normalization) ─────────────────────
hrs_df  = pd.read_csv(os.path.join(OUTPUTS_DIR, 'hrs_risk_categories.csv'))
hrs_raw = hrs_df['HRS_score'].values

# ✅ IMPROVED: Sigmoid normalization (better calibration than linear min-max)
prob_hrs = expit((hrs_raw - hrs_raw.mean()) / hrs_raw.std())
pred_hrs = (hrs_df['risk_class'] == 'High').astype(int).values

print(f'HRS shape     : {hrs_df.shape}')
print(f'Raw range     : [{hrs_raw.min():.3f}, {hrs_raw.max():.3f}]')
print(f'Prob range    : [{prob_hrs.min():.3f}, {prob_hrs.max():.3f}]')
print(f'Risk classes  :\n{hrs_df["risk_class"].value_counts().to_string()}')
print('\n✅ HRS loaded with sigmoid normalization (improved).')

In [ ]:
# ── 5. Evaluate All Models — Table 9 ───────────────────────────────────────
def evaluate(name, y_true, y_pred, y_prob):
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_true, y_pred),                    4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0),  4),
        'Recall'   : round(recall_score(y_true, y_pred, zero_division=0),     4),
        'F1 Score' : round(f1_score(y_true, y_pred, zero_division=0),         4),
        'AUC-ROC'  : round(roc_auc_score(y_true, y_prob),                     4),
        'Brier'    : round(brier_score_loss(y_true, y_prob),                  4),
        'TP':int(tp),'FP':int(fp),'TN':int(tn),'FN':int(fn),
    }

results  = []
roc_data = {}
cm_data  = {}

for name, model in models.items():
    y_pred  = model.predict(X_test)
    y_prob  = model.predict_proba(X_test)[:,1]
    results.append(evaluate(name, y_test, y_pred, y_prob))
    fpr,tpr,_ = roc_curve(y_test, y_prob)
    roc_data[name] = (fpr, tpr, roc_auc_score(y_test, y_prob))
    cm_data[name]  = confusion_matrix(y_test, y_pred)

# Add HRS
results.append(evaluate('HRS', y_test, pred_hrs, prob_hrs))
fpr,tpr,_ = roc_curve(y_test, prob_hrs)
roc_data['HRS'] = (fpr, tpr, roc_auc_score(y_test, prob_hrs))
cm_data['HRS']  = confusion_matrix(y_test, pred_hrs)

table9 = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
table9.to_csv(os.path.join(TABLES_DIR, 'Table9_Model_Performance_Comparison.csv'), index=False)

print('📋 Table 9: Model Performance Comparison')
display(table9[['Model','Accuracy','Precision','Recall','F1 Score','AUC-ROC','Brier']].style
    .background_gradient(subset=['AUC-ROC','F1 Score','Recall'], cmap='YlGn')
    .background_gradient(subset=['Brier'], cmap='YlOrRd_r')
    .format('{:.4f}', subset=['Accuracy','Precision','Recall','F1 Score','AUC-ROC','Brier'])
    .set_caption('Table 9: Model Performance Comparison'))
print('\n✅ Table 9 saved to Tables/Table9_Model_Performance_Comparison.csv')

In [ ]:
# ── 6. Plot 12 — ROC Curves ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.plot([0,1],[0,1],'k--', lw=1.2, label='Random (AUC=0.50)')

all_names = list(models.keys()) + ['HRS']
all_probs = [models[n].predict_proba(X_test)[:,1] for n in models] + [prob_hrs]

for name, (fpr,tpr,auc_val), color in zip(all_names, roc_data.values(), PALETTE):
    ax.plot(fpr, tpr, lw=2.5, color=color, label=f'{name}  (AUC={auc_val:.4f})')

ax.set_xlabel('False Positive Rate (1 − Specificity)', fontsize=11)
ax.set_ylabel('True Positive Rate (Sensitivity)',       fontsize=11)
ax.set_title('Plot 12: ROC Curves — All Models\nHeart Failure Risk Score Prediction',
             fontsize=13, pad=12)
ax.legend(loc='lower right', framealpha=0.9)
ax.set_xlim(0,1); ax.set_ylim(0,1.02)
ax.grid(axis='both', linestyle='--', alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'Plot12_ROC_Curves.png'),
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('✅ Plot 12 saved to Figures/Plot12_ROC_Curves.png')

In [ ]:
# ── 7. Confusion Matrices — All Models ─────────────────────────────────────
all_preds = {n: models[n].predict(X_test) for n in models}
all_preds['HRS'] = pred_hrs

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
cmaps = ['Blues','Greens','Oranges','Purples','Reds']

for ax, (name, pred_), cmap in zip(axes, all_preds.items(), cmaps):
    cm = confusion_matrix(y_test, pred_, labels=[0,1])
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Pred 0','Pred 1'],
                yticklabels=['Actual 0','Actual 1'],
                linewidths=0.5, linecolor='white', cbar=False,
                annot_kws={'size':13,'weight':'bold'})
    tn,fp,fn,tp = cm.ravel()
    acc = accuracy_score(y_test, pred_)
    ax.set_title(f'{name}\nAcc={acc:.3f}  TP={tp} FN={fn}', fontsize=10, pad=8)
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('Actual',    fontsize=9)

fig.suptitle('Confusion Matrices — All Models (Heart Failure Risk Score Prediction)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'Plot12b_Confusion_Matrices.png'),
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('✅ Confusion matrices saved to Figures/Plot12b_Confusion_Matrices.png')

In [ ]:
# ── 8. Classification Reports ───────────────────────────────────────────────
for name, pred_ in all_preds.items():
    print('='*50)
    print(f'  {name}')
    print('='*50)
    print(classification_report(y_test, pred_, target_names=['No Disease','Disease']))

In [ ]:
# ── 9. Summary ─────────────────────────────────────────────────────────────
print('='*55)
print('  ADITYA — ALL OUTPUTS SAVED')
print('='*55)
for folder in ['Tables','Figures']:
    path = os.path.join(os.path.dirname(TABLES_DIR) if folder=='Tables' else os.path.dirname(FIGURES_DIR), folder)
    if os.path.exists(path):
        for f in sorted(os.listdir(path)):
            size = os.path.getsize(os.path.join(path,f))
            print(f'  ✅ {folder}/{f}  ({size:,} bytes)')
print()
best = table9.iloc[0]
print(f'🏆 Best model by AUC-ROC: {best["Model"]} ({best["AUC-ROC"]:.4f})')
print('\n📌 Upload Tables/ and Figures/ to GitHub → raise Pull Request')